In [ ]:
# ============================================================
# COLAB TRANSFER - 01: INSPEÇÃO DO AMBIENTE DE TRANSFERÊNCIA
# ============================================================
# Este notebook roda no GOOGLE COLAB
# Objetivo: Verificar Civitai, Kaggle, dataset e staging
# NÃO configura ComfyUI aqui
# ============================================================

import sys
import subprocess
import json
import os
from pathlib import Path

print("=" * 60)
print("INSPEÇÃO DO AMBIENTE DE TRANSFERÊNCIA (COLAB)")
print("=" * 60)

print(f"Python: {sys.version.split()[0]}")

# Kaggle CLI
try:
    result = subprocess.run(["kaggle", "--version"], capture_output=True, text=True)
    print(f"Kaggle CLI: {result.stdout.strip()}")
except Exception as e:
    print(f"Kaggle CLI: NÃO INSTALADO ({e})")
    print("Instale com: pip install -U kaggle")

# kagglehub
try:
    import kagglehub
    print(f"kagglehub: {kagglehub.__version__}")
except ImportError:
    print("kagglehub: NÃO INSTALADO (opcional, fallback de upload)")

# Verificar autenticação Kaggle (lê de Secrets do Colab via userdata ou arquivo)
def ensure_kaggle_auth():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"
    
    # Se arquivo existe, usa ele
    if kaggle_json.exists():
        with open(kaggle_json) as f:
            creds = json.load(f)
        print(f"Kaggle auth: ENCONTRADO em {kaggle_json}")
        print(f"  Username: {creds.get('username', 'N/A')}")
        return True
    
    # Tenta ler de Secrets do Colab (google.colab.userdata)
    username = None
    key = None
    try:
        from google.colab import userdata
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
    except (ImportError, userdata.NotebookAccessError):
        pass
    
    # Fallback para variáveis de ambiente (outros ambientes)
    if not username:
        username = os.environ.get("KAGGLE_USERNAME")
    if not key:
        key = os.environ.get("KAGGLE_KEY")
    
    if username and key:
        kaggle_dir.mkdir(exist_ok=True)
        kaggle_json.write_text(json.dumps({"username": username, "key": key}))
        os.chmod(kaggle_json, 0o600)
        print(f"Kaggle auth: CRIADO a partir de Secrets do Colab")
        print(f"  Username: {username}")
        return True
    
    print("Kaggle auth: NÃO CONFIGURADO")
    print("  Defina no Colab: Secrets → KAGGLE_USERNAME e KAGGLE_KEY")
    return False

kaggle_ok = ensure_kaggle_auth()

# Variáveis de ambiente Civitai (Colab Secrets via userdata + fallback env)
civitai_token = None
try:
    from google.colab import userdata
    civitai_token = userdata.get('CIVITAI_TOKEN')
except (ImportError, userdata.NotebookAccessError):
    pass
if not civitai_token:
    civitai_token = os.environ.get("CIVITAI_TOKEN") or os.environ.get("CIVITAI_API_KEY")
if civitai_token:
    print(f"Civitai token: CONFIGURADO ({civitai_token[:10]}...)")
else:
    print("Civitai token: NÃO CONFIGURADO")
    print("  Defina no Colab: Secrets → CIVITAI_TOKEN")

# Diretórios de transferência
print("\n=== DIRETÓRIOS DE TRANSFERÊNCIA ===")
STAGING = Path("/content/kaggle_staging")
STAGING.mkdir(parents=True, exist_ok=True)
for d in ["/content", STAGING]:
    p = Path(d)
    print(f"  {d}: {'EXISTE' if p.exists() else 'NÃO EXISTE'}")
    if p.exists():
        files = list(p.iterdir())
        print(f"    Conteúdo: {[f.name for f in files[:10]]}{'...' if len(files) > 10 else ''}")

# Verificar dataset Kaggle
print("\n=== VERIFICANDO DATASET KAGGLE ===")
DATASET = "automamermaid/comfydocs"
result = subprocess.run(
    ["kaggle", "datasets", "metadata", DATASET],
    capture_output=True, text=True
)
print(f"Return code: {result.returncode}")
if result.returncode == 0:
    print("✅ Dataset acessível")
    try:
        metadata = json.loads(result.stdout)
        print(f"  Title: {metadata.get('title')}")
        print(f"  Owner: {metadata.get('ownerSlug')}")
        print(f"  Arquivos no metadata: {len(metadata.get('resources', []))}")
    except:
        print(f"  Metadata raw: {result.stdout[:300]}")
elif "403" in result.stderr:
    print("❌ ERRO 403: Sem permissão no dataset")
    print(f"   STDERR: {result.stderr}")
else:
    print(f"STDOUT: {result.stdout}")
    print(f"STDERR: {result.stderr}")

# Verificar arquivos antigos problemáticos no staging
print("\n=== VERIFICANDO STAGING PARA ARQUIVOS ANTIGOS ===")
old_name = "lustify-v10-krea-turbo-fp8.safetensors"
old_file = STAGING / old_name
if old_file.exists():
    size = old_file.stat().st_size
    if size == 0:
        print(f"⚠️  ENCONTRADO ARQUIVO ANTIGO DE 0 BYTES: {old_name}")
        print(f"   Removendo...")
        old_file.unlink()
        print(f"   ✅ Removido")
    else:
        print(f"⚠️  Arquivo antigo existe com {size} bytes: {old_name}")
else:
    print(f"✅ Nenhum arquivo antigo problemático encontrado")

MODEL_NAME = "lustifyNSFWCheckpoint_v10Krea2.safetensors"
model_file = STAGING / MODEL_NAME
if model_file.exists():
    size = model_file.stat().st_size
    size_gb = size / (1024**3)
    print(f"\n✅ Modelo correto JÁ EXISTE no staging: {size:,} bytes ({size_gb:.2f} GB)")
    EXPECTED_GB = 11.94
    if abs(size_gb - EXPECTED_GB) < 0.5:
        print(f"   Tamanho compatível com esperado (~{EXPECTED_GB} GB)")
    else:
        print(f"   ⚠️  Tamanho DIVERGENTE do esperado ({EXPECTED_GB} GB)")
else:
    print(f"\n📥 Modelo NÃO encontrado no staging, será necessário baixar")

print("\n" + "=" * 60)
print("PRÓXIMO PASSO: Execute 02_download.ipynb")
print("=" * 60)